In [28]:
# ------------------------------------------------------------
# Standard library imports
# ------------------------------------------------------------
import gc
import glob
import logging
import os
import sys
from datetime import datetime

# ------------------------------------------------------------
# Third‑party scientific stack
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import xarray as xr
import xesmf as xe
from auxiliary_functions.time_utils import datetime64_to_yyyymmdd, string_to_yyyymm, convert_time_to_ns, convert_ns_to_datetime, extract_years_months

# ------------------------------------------------------------
# External APIs / data access
# ------------------------------------------------------------
from dateutil.relativedelta import relativedelta

# ------------------------------------------------------------
# Logging configuration
# ------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("data_download.log", mode="a"),
        logging.StreamHandler(sys.stdout),
    ]
)
logger = logging.getLogger(__name__)

# Log uncaught exceptions to file
def log_exception(exc_type, exc_value, exc_traceback):
    if issubclass(exc_type, KeyboardInterrupt):
        # Let Ctrl+C behave normally
        sys.__excepthook__(exc_type, exc_value, exc_traceback)
        return
    logger.error("Uncaught exception", exc_info=(exc_type, exc_value, exc_traceback))

sys.excepthook = log_exception

start_date = "2001-01-01"
end_date = "2010-01-01"
logger.info(f"Start date: {start_date}")
logger.info(f"End date:   {end_date}")

def pressure_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
    # Subset to a specific region and keep only one variable
    subset_data = ds.sel(
        level=pressure_levels,
        time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
    ).assign_coords({'level': pressure_levels.astype(np.int32)})

    target_grid = xr.Dataset(
        {
            "lat": (["lat"], np.arange(-90, 91, 1.0)),
            "lon": (["lon"], np.arange(0, 360, 1.0)),
        }
    )
    regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

    return regridder(subset_data) # type: ignore

yyyymm_strings = pd.date_range(
    pd.to_datetime(start_date).to_period("M").to_timestamp(),
    pd.to_datetime(end_date).to_period("M").to_timestamp(),
    freq="MS"
).strftime("%Y%m")

logger.info(f"Dates: {np.datetime64(start_date).astype('datetime64[h]')} : {np.datetime64(end_date).astype('datetime64[h]')}")

graphcast_data_directory = f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology/"
if not os.path.exists(graphcast_data_directory):
    logger.info("Creating output directory...")
    os.makedirs(graphcast_data_directory, exist_ok=True)
logger.info(f"Graphcast data directory: {graphcast_data_directory}")

years, months = extract_years_months(start_date, end_date)

# Pressure levle variables
pressure_level_base = "/gdex/data/d633000/e5.oper.an.pl"

pressure_levels = xr.DataArray(
    data = [200, 850],
    dims=['level'],
    coords={'level': [200, 850]}
)

pressure_level_variables = {
    "u_component_of_wind": "u",
    "v_component_of_wind": "v",
}

pressure_level_variables_old_names = {
    "u_component_of_wind": "U",
    "v_component_of_wind": "V",
}

for variable in pressure_level_variables.keys():
    # files_list = []

    logger.info(f"{variable}")
    # for ym in yyyymm_strings:
    for year in years:
        logger.info(f"-- {year}")

        if not os.path.exists(f"{graphcast_data_directory}/{year}"):
            os.makedirs(f"{graphcast_data_directory}/{year}", exist_ok=True)
        
        pattern = f"{pressure_level_base}/{year}*/e5.oper.an.pl.*_{pressure_level_variables[variable]}.*.nc"
        files_list = sorted(glob.glob(pattern))

        if not files_list:
            logger.info(f"No files found for variable {variable} in year {year}")
        else:
            logger.info(f"---- Loading files...")
            pressure_level_data = convert_time_to_ns(xr.open_mfdataset(files_list, preprocess=pressure_level_preprocess).load())
            pressure_level_data = pressure_level_data.rename({pressure_level_variables_old_names[variable]: variable})
            if variable == 'v_component_of_wind':
                pressure_level_data = pressure_level_data.sel(level=200, drop=True)

        # logger.info(pressure_level_data)
        logger.info(f"---- Output directory: {graphcast_data_directory}/{year}")
        logger.info(f"---- Saving data...")
        datetimes = pressure_level_data.datetime
        pressure_level_data.to_netcdf(f"{graphcast_data_directory}/{year}/{variable}.nc")
        pressure_level_data.close()
        del pressure_level_data
        gc.collect()

logger.info("Finished")

2026-07-08 12:56:04,182 [INFO] Start date: 2001-01-01
2026-07-08 12:56:04,186 [INFO] End date:   2010-01-01
2026-07-08 12:56:04,190 [INFO] Dates: 2001-01-01T00 : 2010-01-01T00
2026-07-08 12:56:04,191 [INFO] Graphcast data directory: /glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology/
2026-07-08 12:56:04,203 [INFO] u_component_of_wind
2026-07-08 12:56:04,204 [INFO] -- 2001
2026-07-08 12:56:04,213 [INFO] ---- Loading files...
2026-07-08 13:46:16,903 [INFO] ---- Output directory: /glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology//2001
2026-07-08 13:46:16,906 [INFO] ---- Saving data...
2026-07-08 13:46:17,965 [INFO] -- 2002
2026-07-08 13:46:18,008 [INFO] ---- Loading files...
2026-07-08 14:36:00,085 [INFO] ---- Output directory: /glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology//2002
2026-07-08 14:36:00,786 [INFO] ---- Saving data...
2026-07-08 14:36:01,879 [INFO] -- 2003
2026-07-08 14:36:01,923 [INFO] ---- Loading files...


KeyboardInterrupt: 